# v27 — независимый реранкинг и позднее объединение

**Restart Kernel → Run All** в прежнем research `.venv`. 43 сравнения: контроль v25 и 42 смеси расстояний/RRF. Без обучения, GPU, интернета и ограничения времени.

Выбор только на calibration; validation — контроль и один победитель. Кандидаты и порог фиксированы. Это адаптивный development-поиск, не независимый тест. MVP автоматически не меняется.


In [1]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/late_fusion.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import late_fusion as experiment
SOURCE_RUN = 'fusion_extension_v1'
RUN_NAME = 'late_fusion_v1'
ALLOW_OUTER_EVALUATION = True
print('Kernel:', sys.executable, '\nRun:', RUN_NAME, '| experiments:', len(experiment.systems()))
print('CPU, cached features only; обучение: нет; кандидаты и порог R1: фиксированы')


Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: late_fusion_v1 | experiments: 43
CPU, cached features only; обучение: нет; кандидаты и порог R1: фиксированы


## 1. Проверки компонентов


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_late_fusion.py', 'tests/test_fusion_extension.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], check=True)


................................................                         [100%]
48 passed in 6.64s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_late_fusion.py', 'tests/test_fusion_extension.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], returncode=0)

## 2. Проверка источников и фиксация сетки
Исходные bbox, веса, прошлые runs и действующий v25 не изменяются.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Output:', context['output'])
display(context['manifest']['late_fusion_plan'])


Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_27_late_fusion/runs/late_fusion_v1


{'source_directory': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_26_fusion_extension/runs/fusion_extension_v1',
 'source_signature': 'd6003aad015cc8ff7191cde73287c4b1cf64ba2d45e286ae3952649509080136',
 'baseline': {'name': 'r1w50_k20_q3_l50',
  'k1': 20,
  'k2': 3,
  'lambda': 0.5,
  'r1_weight': 0.5},
 'graphs': 'independent MVP and R1 static galleries; k1=20, k2=3; MVP lambda=.5, R1=.5/.75',
 'fusion': 'distance mean or weighted reciprocal rank with k=10/60; full gallery before top10',
 'candidate': 'unchanged raw R1 cosine, candidate and frozen threshold',
 'selection': 'maximum calibration mAP; ties retain v25 first',
 'validation': 'v25 control and one calibration winner only; no automatic promotion',
 'scope': 'adaptive search on observed development splits, not an independent hidden test',
 'rrf_source': 'https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf',
 'threshold_fit': False,
 'encoder_forwards': 0,
 'o

## 3. Calibration → замороженный выбор → validation
Прогресс и время показываются для каждого сравнения. Прерванный запуск продолжится с тем же RUN_NAME.


In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
print('Status:', results['status'], '| promoted:', results['promoted'])



CALIBRATION 1/43 | elapsed 10.5s
[START] calibration_V25_control | elapsed 0.00 h
calibration V25_control: mAP=0.826410; R1 candidates unchanged
[DONE] calibration_V25_control | 0.0 min

CALIBRATION 2/43 | elapsed 10.8s
[START] calibration_distance_r1w10_l50 | elapsed 0.00 h
calibration distance_r1w10_l50: mAP=0.795105; R1 candidates unchanged
[DONE] calibration_distance_r1w10_l50 | 0.0 min

CALIBRATION 3/43 | elapsed 11.2s
[START] calibration_distance_r1w25_l50 | elapsed 0.00 h
calibration distance_r1w25_l50: mAP=0.807423; R1 candidates unchanged
[DONE] calibration_distance_r1w25_l50 | 0.0 min

CALIBRATION 4/43 | elapsed 11.6s
[START] calibration_distance_r1w40_l50 | elapsed 0.00 h
calibration distance_r1w40_l50: mAP=0.814767; R1 candidates unchanged
[DONE] calibration_distance_r1w40_l50 | 0.0 min

CALIBRATION 5/43 | elapsed 12.0s
[START] calibration_distance_r1w50_l50 | elapsed 0.00 h
calibration distance_r1w50_l50: mAP=0.824546; R1 candidates unchanged
[DONE] calibration_distance_r

## 4. Результаты


In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Reports and exports:', context['output'])


# v27 — объединение после независимого реранкинга

Статус: complete
43 фиксированных варианта: контроль v25 и 42 смеси distance/RRF10/RRF60.
Доля R1: 10/25/40/50/60/75/90%; λ R1: 0.50/0.75; оба графа 20/3, λ MVP=0.50.
Выбор только на calibration; при равенстве сохраняется v25.

| Validation: v25 и выбор | mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|
| V25_control | 0.828957 | 0.829960 | 0.795455 | 0.709677 |
| distance_r1w75_l75 | 0.818643 | 0.817814 | 0.795455 | 0.709677 |

ΔmAP против v25: -0.010315. AP лучше/хуже/равно: 19/29/199.

## Calibration

| Вариант | mAP@10 |
|---|---:|
| distance_r1w75_l75 | 0.831855 |
| distance_r1w75_l50 | 0.831014 |
| distance_r1w90_l50 | 0.830369 |
| rrf60_r1w90_l50 | 0.829691 |
| distance_r1w60_l50 | 0.827624 |
| rrf10_r1w90_l50 | 0.827295 |
| distance_r1w60_l75 | 0.826717 |
| distance_r1w90_l75 | 0.826553 |
| V25_control | 0.826410 |
| rrf60_r1w90_l75 | 0.825087 |
| rrf10_r1w75_l50 | 0.824939 |
| distance_r1w50_l50 | 0.824546 |
| rrf10_r1w75_l75 | 0.823558 |
| rrf10_r1w90_l75 | 0.823086 |
| rrf60_r1w75_l50 | 0.822125 |
| rrf60_r1w75_l75 | 0.821674 |
| rrf10_r1w60_l50 | 0.817557 |
| rrf10_r1w60_l75 | 0.816867 |
| distance_r1w50_l75 | 0.815480 |
| distance_r1w40_l50 | 0.814767 |
| rrf60_r1w60_l75 | 0.813127 |
| rrf60_r1w60_l50 | 0.812554 |
| distance_r1w40_l75 | 0.810150 |
| rrf10_r1w50_l50 | 0.809243 |
| rrf10_r1w50_l75 | 0.808556 |
| distance_r1w25_l50 | 0.807423 |
| rrf60_r1w50_l75 | 0.806586 |
| rrf60_r1w50_l50 | 0.805917 |
| rrf10_r1w40_l50 | 0.802334 |
| rrf10_r1w40_l75 | 0.802040 |
| rrf60_r1w40_l75 | 0.801122 |
| distance_r1w25_l75 | 0.800744 |
| rrf60_r1w40_l50 | 0.800417 |
| distance_r1w10_l50 | 0.795105 |
| distance_r1w10_l75 | 0.794489 |
| rrf10_r1w25_l50 | 0.793151 |
| rrf10_r1w25_l75 | 0.792804 |
| rrf60_r1w25_l75 | 0.792498 |
| rrf60_r1w25_l50 | 0.791163 |
| rrf10_r1w10_l50 | 0.788543 |
| rrf60_r1w10_l75 | 0.788484 |
| rrf60_r1w10_l50 | 0.788155 |
| rrf10_r1w10_l75 | 0.788114 |

## Ограничения

Это адаптивный поиск на уже наблюдавшихся development-данных, не независимая оценка hidden test.
Validation — только контроль v25 и один calibration-победитель. Автоматического продвижения нет.
Порог, кандидаты, confidence, эмбеддинги, bbox и evaluator неизменны; другие query недоступны scorer.
RRF использует позиции во всей gallery, с единицы; это взвешенная адаптация, не обещание выигрыша.
[Источник RRF](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf).
Источники неизменны: True. Время текущего вызова: 0.5 мин.


Reports and exports: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_27_late_fusion/runs/late_fusion_v1
